##KPI_1

In [0]:
select * from gold_workforce.azure_blob_storage.fact_general_ledgers;
--select * from gold_workforce.azure_blob_storage.fact_payroll;

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_1 AS
SELECT
  date_trunc('month', posting_date) AS month,
  SUM(total_amount) AS total_revenue,
  ROUND(
    100 * (SUM(total_amount) - LAG(SUM(total_amount)) OVER (ORDER BY date_trunc('month', posting_date)))
    / NULLIF(LAG(SUM(total_amount)) OVER (ORDER BY date_trunc('month', posting_date)), 0), 2
  ) AS mom_growth_pct
FROM gold_workforce.azure_blob_storage.fact_general_ledgers
GROUP BY date_trunc('month', posting_date)
ORDER BY month

##KPI_2

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_2 AS
SELECT
  gl.*,
  da.account_code,
  da.account_name,
  da.account_type,
  da.category,
  da.is_active
FROM gold_workforce.azure_blob_storage.fact_general_ledgers gl
JOIN gold_workforce.azure_blob_storage.dim_accounts da
  ON gl.account_id = da.account_id

In [0]:
SELECT
  date_trunc('month', gl.posting_date) AS month,
  gl.company_id,
  SUM(gl.total_amount) AS cost_of_sales
FROM gold_workforce.azure_blob_storage.fact_general_ledgers gl
JOIN gold_workforce.azure_blob_storage.dim_accounts da
  ON gl.account_id = da.account_id
WHERE da.account_type = 'COGS' and da.account_type = 'Expense'
GROUP BY month, gl.company_id
ORDER BY month, gl.company_id

##KPI_5

In [0]:
SELECT
  e.*,
  p.*
FROM gold_workforce.azure_blob_storage.dim_employee e
JOIN gold_workforce.azure_blob_storage.fact_payroll p
  ON e.employee_id = p.employee_id
join gold_workforce.azure_blob_storage.dim_company c
  ON p.company_id = c.company_id

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_5 AS

SELECT
  e.position,
  c.company_id,
  AVG(p.base_salary + p.bonus + p.overtime_pay + p.commission) AS avg_total_compensation
FROM gold_workforce.azure_blob_storage.dim_employee e
JOIN gold_workforce.azure_blob_storage.fact_payroll p
  ON e.employee_id = p.employee_id
join gold_workforce.azure_blob_storage.dim_company c
  ON p.company_id = c.company_id
GROUP BY e.position, c.company_id
ORDER BY e.position, c.company_id

##KPI_6

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_6 AS
SELECT
  date_trunc('month', gl.posting_date) AS month,
  SUM(gl.total_amount) AS net_profit
FROM gold_workforce.azure_blob_storage.fact_general_ledgers gl
JOIN gold_workforce.azure_blob_storage.dim_accounts da
  ON gl.account_id = da.account_id
WHERE da.category IN ('Revenue', 'Cost of Sales', 'Operating Expenses', 'Other Income', 'Other Expenses')
GROUP BY date_trunc('month', gl.posting_date)
ORDER BY month

##KPI_7

In [0]:
SELECT
  d.department_name,
  SUM(p.overtime_pay) AS total_overtime_pay,
  SUM(p.bonus) AS total_bonus,
  SUM(p.overtime_pay + p.bonus) AS total_variable_compensation,
  SUM(p.overtime_pay) / NULLIF(SUM(p.base_salary), 0) AS overtime_to_base_salary_ratio
FROM gold_workforce.azure_blob_storage.dim_employee e
JOIN gold_workforce.azure_blob_storage.fact_payroll p
  ON e.employee_id = p.employee_id
JOIN gold_workforce.azure_blob_storage.dim_departments d
  ON p.department_id = d.department_id
GROUP BY d.department_name
ORDER BY d.department_name

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_7 AS
SELECT
  d.department_name,
  SUM(p.base_salary + p.bonus + p.overtime_pay + p.commission) AS total_payroll_expenses
FROM gold_workforce.azure_blob_storage.dim_departments d
LEFT JOIN gold_workforce.azure_blob_storage.fact_payroll p
  ON d.department_id = p.department_id
LEFT JOIN gold_workforce.azure_blob_storage.dim_employee e
  ON p.employee_id = e.employee_id
LEFT JOIN gold_workforce.azure_blob_storage.fact_general_ledgers gl
  ON d.department_id = gl.department_id
LEFT JOIN gold_workforce.azure_blob_storage.dim_accounts da
  ON gl.account_id = da.account_id
GROUP BY d.department_name
ORDER BY d.department_name

##KPI_8

In [0]:
select * from gold_workforce.azure_blob_storage.dim_employee

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_8 AS
SELECT
  d.department_name,
  COUNT(e.employee_id) AS active_employee_count
FROM gold_workforce.azure_blob_storage.dim_employee e
JOIN gold_workforce.azure_blob_storage.dim_departments d
  ON e.department_id = d.department_id
WHERE e.is_active = true
GROUP BY d.department_name
ORDER BY d.department_name

##KPI_9

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_9 AS
SELECT
  d.department_name,
  COUNT(e.employee_id) AS active_employee_count
FROM gold_workforce.azure_blob_storage.dim_departments d
JOIN gold_workforce.azure_blob_storage.dim_employee e
  ON d.department_id = e.department_id
WHERE e.is_active = true
GROUP BY d.department_name
ORDER BY d.department_name

##KPI_10

In [0]:
CREATE or replace VIEW gold_workforce.kpis.kpi_10 AS
SELECT
  c.company_name,
  SUM(p.base_salary + p.bonus + p.overtime_pay + p.commission) AS total_payroll_cost,
  SUM(CASE WHEN da.category = 'Operating Revenue' THEN gl.total_amount ELSE 0 END) AS total_company_revenue,
  SUM(p.base_salary + p.bonus + p.overtime_pay + p.commission) / NULLIF(SUM(CASE WHEN da.category = 'Operating Revenue' THEN gl.total_amount ELSE 0 END), 0) AS payroll_cost_pct_of_revenue
FROM gold_workforce.azure_blob_storage.dim_company c
JOIN gold_workforce.azure_blob_storage.fact_payroll p
  ON c.company_id = p.company_id
JOIN gold_workforce.azure_blob_storage.fact_general_ledgers gl
  ON c.company_id = gl.company_id
JOIN gold_workforce.azure_blob_storage.dim_accounts da
  ON gl.account_id = da.account_id
GROUP BY c.company_name
ORDER BY c.company_name

In [0]:
select * from gold_workforce.azure_blob_storage.data_cube